# JPX 东京证券交易所预测 — 探索性数据分析

本文档对核心数据进行全面探索，包括：
1. 数据概览与基本统计
2. 缺失值分析
3. Target 分布与统计特征
4. 行业/市值分布
5. 价格走势与自相关性

In [ ]:
import sys
sys.path.append('..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.data.loader import load_merged
from src.data.preprocessor import preprocess

plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei']
plt.rcParams['axes.unicode_minus'] = False

%matplotlib inline

## 1. 数据概览

In [ ]:
df = load_merged(use_train=True, use_supplement=False)
print(f'数据形状: {df.shape}')
print(f'日期范围: {df["Date"].min().date()} ~ {df["Date"].max().date()}')
print(f'股票数量: {df["SecuritiesCode"].nunique()}')
print(f'交易日数量: {df["Date"].nunique()}')
df.head()

In [ ]:
df.describe()

## 2. 缺失值分析

In [ ]:
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_df = pd.DataFrame({'缺失数': missing, '缺失比例%': missing_pct})
missing_df = missing_df[missing_df['缺失数'] > 0].sort_values('缺失比例%', ascending=False)
print(missing_df)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
missing_df['缺失比例%'].plot.bar(ax=ax)
ax.set_title('各列缺失值比例')
ax.set_ylabel('缺失比例 (%)')
plt.tight_layout()
plt.show()

## 3. Target 分布

In [ ]:
target = df['Target'].dropna()
print(f'均值: {target.mean():.6f}')
print(f'标准差: {target.std():.6f}')
print(f'偏度: {target.skew():.4f}')
print(f'峰度: {target.kurtosis():.4f}')
print(f'最小值: {target.min():.6f}')
print(f'最大值: {target.max():.6f}')
print(f'中位数: {target.median():.6f}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].hist(target, bins=200, edgecolor='black', alpha=0.7)
axes[0].set_title('Target 分布（全范围）')
axes[0].set_xlabel('Target')
axes[0].set_ylabel('频次')

clipped = target.clip(upper=target.quantile(0.99), lower=target.quantile(0.01))
axes[1].hist(clipped, bins=200, edgecolor='black', alpha=0.7, color='orange')
axes[1].set_title('Target 分布（1%-99%截断）')
axes[1].set_xlabel('Target')

plt.tight_layout()
plt.show()

In [ ]:
daily_mean = df.groupby('Date')['Target'].mean()
fig, ax = plt.subplots(figsize=(16, 5))
ax.plot(daily_mean.index, daily_mean.values, linewidth=0.5)
ax.set_title('日均 Target 随时间变化')
ax.set_xlabel('日期')
ax.set_ylabel('日均 Target')
ax.axhline(y=0, color='red', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

## 4. 行业分布

In [ ]:
sector_counts = df.groupby('33SectorName')['SecuritiesCode'].nunique().sort_values(ascending=False)
fig, ax = plt.subplots(figsize=(12, 6))
sector_counts.plot.bar(ax=ax)
ax.set_title('各行业股票数量')
ax.set_ylabel('股票数量')
plt.tight_layout()
plt.show()

In [ ]:
sector_target = df.groupby('33SectorName')['Target'].agg(['mean', 'std']).sort_values('mean', ascending=False)
fig, ax = plt.subplots(figsize=(14, 5))
ax.bar(range(len(sector_target)), sector_target['mean'], 
        yerr=sector_target['std']*0.1, capsize=2)
ax.set_xticks(range(len(sector_target)))
ax.set_xticklabels(sector_target.index, rotation=90)
ax.set_title('各行业平均 Target（误差条=0.1*标准差）')
ax.axhline(y=0, color='red', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

## 5. 市值分布与Universe

In [ ]:
print(f'Universe0=True 的股票数: {df[df["Universe0"]==True]["SecuritiesCode"].nunique()}')
print(f'Universe0=False 的股票数: {df[df["Universe0"]==False]["SecuritiesCode"].nunique()}')

## 6. 自相关性分析

In [ ]:
from statsmodels.graphics.tsaplots import plot_acf

sample_code = df['SecuritiesCode'].value_counts().index[0]
stock_data = df[df['SecuritiesCode'] == sample_code].sort_values('Date')

fig, ax = plt.subplots(figsize=(10, 4))
plot_acf(stock_data['Target'].dropna(), lags=20, ax=ax)
ax.set_title(f'股票 {sample_code} 的 Target 自相关图')
plt.tight_layout()
plt.show()